In [ ]:
# general imports 
import numpy as np
import os
import matplotlib.pyplot as plt
import math
import pandas as pd
import MDAnalysis as mda
from IPython.display import Video

# Today's Practical

- We will make a step forward toward biological simulations.
- The main idea of today's practical is to see how principal physics behind intermolecular interactions contributes to the formation of biological complexes such as lipid membranes.

In [ ]:
!ls -l

## What is inside?

### force_field

Now you know what it is. :)

### images

Images related to the practical.

### scripts

Contains `.mdp` files we will check and discuss today.

### structures

Starting atomic configurations of molecules of interest.

### topologies

`.itp` files for preparing/performing simulations of todays' practical.

### trajectories

Contains long, already conducted simulations for the purposes of today's practical.

### videos

Videos related to the practical.

### week4.ipynb

Jupyter notebook we are using right now.

### Create a folder where we will be working today

In [ ]:
!mkdir playground/

In [ ]:
%cd playground/

In [ ]:
!mkdir ions/

In [ ]:
%cd ions/

## First part: Ions in water

### Role of ions in biology

- Ions are everywhere.
- All biological fluids containt them.
- Multiple roles:
    - Contributing to cellular membrane potential.
    - Participating in neurotransmission.
    - Involved in cell signaling.
    - Maiintaning the osmotic balance.
    - And many more...

### Who knows what is NaCl?

### Preparing the system

### `gmx insert-molecules`

- `gmx insert-molecules` is designed to insert molecules into existing structures or simply empty boxes.
- `-ci` &mdash; `.gro` structure to insert
- `-f` &mdash; (optional) `.gro` structure into which to insert
- `-nmol` &mdash; number of molecules to insert
- `-o` &mdash; name of the output `.gro` structure
- `-box` &mdash; define the box size if you do not have any initial structure to insert into (defined by `-f`)
- `-try` &mdash; an attempt to insert the structure could be unsuccessful because of an overlap with already present molecules or/and not enough available free space; therefore, we should specify total number of possible attempts
- `-replace` &mdash; which molecules you can replace upon inserting
- there are other options that we will not cover today; check `gmx insert-molecules -h` or https://manual.gromacs.org/current/onlinehelp/gmx-insert-molecules.html yourself

In [ ]:
!gmx insert-molecules -ci ../../structures/Na.gro -o Na.gro -nmol 200 -box 2 2 2 -try 2000

### What did we do?

- We inserted 200 sodium ions into an empty box with a cubic size 2 nm.

In [ ]:
!gmx insert-molecules -ci ../../structures/Cl.gro -f Na.gro -o Na_Cl.gro -nmol 200 -try 2000

### What did we do?

- We inserted additionally 200 chloride ions into just created box with sodium cations.

In [ ]:
!gmx editconf -f Na_Cl.gro -o ions.gro -box 6 6 6

### What did we do?

- We changed the box size from 2 nm to 6 nm.
- Now we have a "crystal" (although it's not true, simply a piece) of NaCl in the middle of the box.

<img src='../../images/box_with_ions_before.png' width=600 height=450 alt='box_with_ions_before'>

In [ ]:
!gmx solvate -cp ions.gro -cs ../../structures/water_three_site.gro -o ions_in_water.gro -maxsol 6000

### What did we do?

- We solvated NaCl in 6000 water molecules (three-site model).
- You didn't forget about three-site, four-site, etc. models, right?

<img src='../../images/box_with_ions.png' width=600 height=450 alt='box_with_ions'>

### Writing topology

In [ ]:
topology = """; force field parameters
#include "../../force_field/charmm36-jul2022.ff/forcefield.itp"
#include "../../force_field/charmm36-jul2022.ff/tip3p.itp"
#include "../../force_field/charmm36-jul2022.ff/ions.itp"

[ system ]
; name of the system
200 NaCl ion pairs in 6000 waters

[ molecules ]
; compound   #mols
NA           200
CL           200
SOL          6000
"""

with open('topol.top', 'w') as outFile:
    outFile.write(topology)

### Let's check force field files more carefully today
- We know more now.

### Topology of water

In [ ]:
!cat ../../force_field/charmm36-jul2022.ff/tip3p.itp

### Topology of ions

In [ ]:
!head -n 50 ../../force_field/charmm36-jul2022.ff/ions.itp

### Parameters for nonbonded interactions

- Different atom types have different parameters.
- Atom types used for a specific molecule given in the topology as we showed above.

In [ ]:
!sed -n '51,100p' ../../force_field/charmm36-jul2022.ff/ffnonbonded.itp

### Back to simulations

### Minimizing the system
- Recalling week02.

In [ ]:
!gmx grompp -p topol.top -c ions_in_water.gro -f ../../scripts/minim.mdp -o minim.tpr -po minim_mdout.mdp

In [ ]:
!gmx mdrun -deffnm minim -v

### Cleaning temporary files (if any)

In [ ]:
!rm *step*

### Checking the potential energy
- Recalling week02.

In [ ]:
!echo 'Potential' | gmx energy -f minim.edr -o pot_minim.xvg

In [ ]:
potential = np.loadtxt('pot_minim.xvg', comments=['#', '@'])
plt.plot(potential[:, 0], potential[:, 1], 'r*')
plt.legend(['minimization'])
plt.xlabel('Iteration')
plt.ylabel('Potential Energy  (kJ/mol)')
plt.show()

### Equilibration and production

In [ ]:
# !gmx grompp -maxwarn 2 -p topol.top -c minim.gro -f ../../scripts/eq.mdp -o eq.tpr -po eq_mdout.mdp

In [ ]:
# !gmx mdrun -deffnm eq -v

In [ ]:
# !gmx grompp -p topol.top -c eq.gro -f ../../scripts/prod.mdp -o prod.tpr -po prod_mdout.mdp

In [ ]:
# !gmx mdrun -deffnm prod -v

### These simulations are a bit costly
- We will not run these simulations now (commands are commented).
- They were already performed for you.
- Let's copy the trajectories in the current directory.

In [ ]:
!cp ../../trajectories/ions/* .

### More discussion about `.mdp` file

In [ ]:
!cat ../../scripts/eq.mdp

In [ ]:
Video('../../videos/box_with_ions_eq.mp4', width=800, height=600, embed=True)

## Before and after

<img src='../../images/box_with_ions_before.png' width=600 height=450 alt='box_with_ions_before'>
<img src='../../images/box_with_ions_after.png' width=600 height=450 alt='box_with_ions_after'>

### Dissolution of the cluster
- The ionic cluster dissolved within 10-15 ns.
- The time scale is force-field dependent but multiple studies exist that focus on dissolution (in tandem with experimental techniques).

### Some numerical analysis

### `gmx mindist` 

- `gmx mindist` computes the distance and number of contacts between two atom groups.
- `-f` &mdash; input trajectory, `.xtc` in our case
- `-s` &mdash; (optional) `.tpr` file that has default atom groups in case you didn't prepare `.ndx` file
- `-n` &mdash; `.ndx` file with defined atom groups
- `-b` &mdash; simulation time to start analysis (useful when you want to analyze only a fragment of your simulation)
- `-e` &mdash; simulation time to end analysis (useful when you want to analyze only a fragment of your simulation)
- `-d` &mdash; distance for contacts in nm
- there are other options that we will not cover today; check `gmx mindist -h` or https://manual.gromacs.org/current/onlinehelp/gmx-mindist.html yourself

### Number of contacts
- We will calculate number of contacts of all ions with each other.
- `Ion` is a default group, i.e., no need to create `.ndx` file.

In [ ]:
input_command = """Ion
Ion
"""
!echo "$input_command" | gmx mindist -f eq.xtc -s eq.tpr -d 0.5 -od mindist_eq.xvg -on numcont_eq.xvg

In [ ]:
contacts_eq = np.loadtxt('numcont_eq.xvg', comments=['#', '@'])
plt.plot(contacts_eq[:, 0], contacts_eq[:, 1], 'r*')
plt.legend(['equilibration'])
plt.xlabel('Time (ps)')
plt.ylabel('Number of ion contacts')
plt.show()

### TASK
- RDF of NaCl from the production run.
- Recall of previous week.
- Commented template below.
- Link to the previous week https://gitlab.com/denysbiriukov/survival_guide_simulations_c9555/-/blob/main/week03/week03.ipynb?ref_type=heads

### Creating index file
- We should create separate groups for sodium and chloride.

In [ ]:
# input_command = """a {ADD_HERE_WHAT_NEEDED_#1}
# a {ADD_HERE_WHAT_NEEDED_#2}
# q
# """
# !echo "$input_command" | gmx make_ndx -f prod.gro -o index.ndx

### Calculating RDF

In [ ]:
# !gmx rdf -f prod.xtc -n index.ndx -o rdf_NA_CL_pure_water.xvg -ref {ADD_HERE_WHAT_NEEDED_#1} -sel {ADD_HERE_WHAT_NEEDED_#2}

## MDAnalysis: second try

- Analyses that are already tested with GROMACS:
    - https://userguide.mdanalysis.org/stable/examples/analysis/structure/average_rdf.html
- Let's try RDF.

In [ ]:
from MDAnalysis.analysis import contacts

u = mda.Universe('eq.gro', 'eq.xtc')

IONS = u.select_atoms('name NA CL')

contacts_eq_mda = []
for ts in u.trajectory:
    distances = contacts.distance_array(IONS.positions, IONS.positions, u.dimensions)
    contacts_calc = contacts.contact_matrix(distances, 5).sum()
    contacts_eq_mda.append([ts.frame, contacts_calc])

contacts_eq_mda_df = pd.DataFrame(contacts_eq_mda, columns=['Frame', 'Contacts'])

In [ ]:
plt.plot(contacts_eq_mda_df['Frame'], contacts_eq_mda_df['Contacts'], 'b*')
plt.legend(['equilibration'])
plt.xlabel('Frame number')
plt.ylabel('Number of ion contacts')
plt.show()

### Difference between results from GROMACS and MDAnalysis
- Plot together both.
- Try to explain the difference if any.

### TASK
- RDF of NaCl from the production run but now using MDAnalysis.
- Recall of previous week.
- Commented template below.
- Link to the previous week https://gitlab.com/denysbiriukov/survival_guide_simulations_c9555/-/blob/main/week03/week03.ipynb?ref_type=heads

In [ ]:
# from MDAnalysis.analysis import rdf
# 
# u = mda.Universe('{ADD_HERE_WHAT_NEEDED}', '{ADD_HERE_WHAT_NEEDED}')
# 
# NA = u.select_atoms('{ADD_HERE_WHAT_NEEDED}')
# CL = u.select_atoms('{ADD_HERE_WHAT_NEEDED}')
# rdf_NA_CL_mda = rdf.InterRDF(NA, CL, bins=100)
# rdf_NA_CL_mda.run()

In [ ]:
# plt.plot(rdf_NA_CL_mda.results.bins, rdf_NA_CL_mda.results.rdf, 'b-')
# plt.legend(['RDF NA-CL MDA'])
# plt.xlabel('r (A)')
# plt.ylabel('g (r)')
# plt.show()

### Difference between results from GROMACS and MDAnalysis
- Plot together both.
- Try to explain the difference if any.

### Cluster analysis
- Apparently, we talk about a cluster of ions here, so for this there is a dedicated tool in GROMACS.

### `gmx clustsize` 

- `gmx clustsize` calculates the number and size of clusters in your systems.
- `-f` &mdash; input trajectory, `.xtc` in our case
- `-n` &mdash; (mandatory) `.ndx` file with defined atom groups
- `-s` &mdash; (optional) `.tpr` file
- `-b` &mdash; simulation time to start analysis (useful when you want to analyze only a fragment of your simulation)
- `-e` &mdash; simulation time to end analysis (useful when you want to analyze only a fragment of your simulation)
- `-cut` &mdash; largest distance to be considered in a cluster
- there are other options that we will not cover today; check `gmx clustsize -h` or https://manual.gromacs.org/current/onlinehelp/gmx-clustsize.html yourself

In [ ]:
input_command = """1
"""
!echo "$input_command" | gmx clustsize -f eq.xtc -n index.ndx -cut 0.5 -nc nclust_eq.xvg -mc maxclust_eq.xvg

### What does 1 in input mean?

### Generated outputs
- Multiple outputs are generated by default.
- Today we are interested in two which we named explicitly.
- `nclust_eq.xvg` &mdash; number of clusters as the function of time
- `maxclust_eq.xvg` &mdash; maximum size of a cluster as the function of time

In [ ]:
maxclust_eq = np.loadtxt('maxclust_eq.xvg', comments=['#', '@'])
plt.plot(maxclust_eq[:, 0], maxclust_eq[:, 1], 'r*')
plt.legend(['equilibration'])
plt.xlabel('Time (ps)')
plt.ylabel('Maximum number of species in a cluster')
plt.show()

### Explain the results

### TASK
- Plot the number of clusters.

## Second part: Lipids in water

In [ ]:
%cd ..

In [ ]:
!mkdir lipids/

In [ ]:
%cd lipids/

### Why lipids?

- Lipids are the major component of cellular membranes.
- They have two distinct parts: headgroup and lipid tails.
- Headgroup is hydrophilic, tails are hydrophobic.
- Therefore, once solvated, hydrophopic parts want to be close to each other.
- In clever terms, above the critical micellar concentration, lipids will form micelles.
- If the concentration is even higher, the lamerral phase will be formed.

<img src='../../images/DPPC_lipid.png' width=600 height=450 alt='DPPC_lipid'>
<img src='../../images/lipid_assemblies.png' width=600 height=450 alt='lipid_assemblies'>

In [ ]:
!gmx insert-molecules -ci ../../structures/DPPC.gro -o lipids.gro -nmol 20 -box 6 6 6 -try 2000

### What did we do?

- We inserted 20 lipid molecules into an empty box with a cubic size 6 nm.

In [ ]:
!gmx solvate -cp lipids.gro -cs ../../structures/water_three_site.gro -o lipids_in_water.gro -maxsol 2000

### What did we do?

- We solvated the box with 2000 water molecules.
- Therefore, we have 100 water molecules per one lipid.

<img src='../../images/box_with_lipids.png' width=600 height=450 alt='box_with_lipids'>

In [ ]:
topology = """; force field parameters
#include "../../force_field/charmm36-jul2022.ff/forcefield.itp"
#include "../../force_field/charmm36-jul2022.ff/tip3p.itp"
#include "../../force_field/charmm36-jul2022.ff/ions.itp"
#include "../../topologies/DPPC.itp"

[ system ]
; name of the system
20 DPPC lipids in 2000 water

[ molecules ]
; compound   #mols
DPPC         20
SOL          2000
"""

with open('topol.top', 'w') as outFile:
    outFile.write(topology)

### Checking the force field file again

In [ ]:
!head -n 50 ../../topologies/DPPC.itp

In [ ]:
!sed -n '150,200p' ../../topologies/DPPC.itp

### What is different?
- There are no force field parameters right here.
- Only connectivity is defined.

In [ ]:
!sed -n '51,100p' ../../force_field/charmm36-jul2022.ff/ffbonded.itp

### Back to simulations

### Minimizing the system
- Recalling week02.

In [ ]:
!gmx grompp -p topol.top -c lipids_in_water.gro -f ../../scripts/minim.mdp -o minim.tpr -po minim_mdout.mdp

In [ ]:
!gmx mdrun -deffnm minim -v

### Cleaning temporary files (if any)

In [ ]:
!rm *step*

### Checking the potential energy
- Recalling week02.

In [ ]:
!echo 'Potential' | gmx energy -f minim.edr -o pot_minim.xvg

In [ ]:
potential = np.loadtxt('pot_minim.xvg', comments=['#', '@'])
plt.plot(potential[:, 0], potential[:, 1], 'r*')
plt.legend(['minimization'])
plt.xlabel('Iteration')
plt.ylabel('Potential Energy  (kJ/mol)')
plt.show()

### Equilibration and production

In [ ]:
# !gmx grompp -maxwarn 2 -p topol.top -c minim.gro -f ../scripts/eq.mdp -o eq.tpr -po eq_mdout.mdp

In [ ]:
# !gmx mdrun -deffnm eq -v

In [ ]:
# gmx grompp -p topol.top -c eq.gro -f ../scripts/prod.mdp -o prod.tpr -po prod_mdout.mdp

In [ ]:
# !gmx mdrun -deffnm prod -v

### Again, these simulations are a bit costly
- We will not run these simulations now (commands are commented).
- They were already performed for you.
- Let's copy the trajectories in the current directory.

In [ ]:
!cp ../../trajectories/lipids/* .

### Quick check of equilibration

In [ ]:
Video('../../videos/box_with_lipids_eq.mp4', width=800, height=600, embed=True)

### Checking the volume

In [ ]:
!echo 'Volume' | gmx energy -f eq.edr -o vol_eq.xvg

In [ ]:
volume_eq = np.loadtxt('vol_eq.xvg', comments=['#', '@'])
plt.plot(volume_eq[:, 0], volume_eq[:, 1], 'r*')
plt.legend(['equilibration'])
plt.xlabel('Time (ps)')
plt.ylabel('Volume ($\mathrm{nm^{3}}$)')
plt.show()

### Analysis of the production simulation
- Volume should be stable.

In [ ]:
!echo 'Volume' | gmx energy -f prod.edr -o vol_prod.xvg

In [ ]:
volume_prod = np.loadtxt('vol_prod.xvg', comments=['#', '@'])
plt.plot(volume_prod[:, 0], volume_prod[:, 1], 'r*')
plt.legend(['production'])
plt.xlabel('Time (ps)')
plt.ylabel('Volume ($\mathrm{nm^{3}}$)')
plt.show()

### Number of contacts from production simulation
- We will calculate number of contacts of all lipid atoms with each other.
- DPPC is recognized, i.e., all lipids atoms are assigned to this group.

In [ ]:
input_command = """DPPC
DPPC
"""
!echo "$input_command" | gmx mindist -f prod.xtc -s prod.tpr -d 0.35 -od mindist_prod.xvg -on numcont_prod.xvg

In [ ]:
contacts_prod = np.loadtxt('numcont_prod.xvg', comments=['#', '@'])
plt.plot(contacts_prod[:, 0], contacts_prod[:, 1], 'r*')
plt.legend(['production'])
plt.xlabel('Time (ps)')
plt.ylabel('Number of lipid contacts')
plt.show()

### We have a micelle with sort of stable number of contacts

### TASK
- Calculate number of contacts from an equilibrium simulation.
- Compare to the one in the production simulation.
- What do you expect to see there?

### Solvent accessible surface area

- A measure of the surface area of a molecule(s) that is accessible to a solvent molecule.

### `gmx sasa`

- `-f` &mdash; input trajectory, `.xtc` in our case
- `-n` &mdash; (optional) `.ndx` file with defined atom groups
- `-s` &mdash; (optional) `.tpr` file with predefined atom groups
- `-b` &mdash; simulation time to start analysis (useful when you want to analyze only a fragment of your simulation)
- `-e` &mdash; simulation time to end analysis (useful when you want to analyze only a fragment of your simulation)
- `-surface` &mdash; selection for surface calculation
- there are other options that we will not cover today; check `gmx sasa -h` or https://manual.gromacs.org/current/onlinehelp/gmx-sasa.html yourself

In [ ]:
!gmx sasa -f eq.xtc -s eq.tpr -o sasa_eq.xvg -surface DPPC

In [ ]:
sasa_eq = np.loadtxt('sasa_eq.xvg', comments=['#', '@'])
plt.plot(sasa_eq[:, 0], sasa_eq[:, 1], 'r*')
plt.legend(['production'])
plt.xlabel('Time (ps)')
plt.ylabel('Area ($\mathrm{nm^{2}}$)')
plt.show()

### TASK
- Calculate and plot SASA from a production simulation.

### BONUS TASK

During last three weeks, you learned following tools for the analysis of MD simulations:

- `gmx energy`
- `gmx density`
- `gmx rdf`
- `gmx msd`
- `gmx mindist`

Related tools include:

- `gmx analyze` &mdash; for estimating the error.
- `gmx make_ndx` &mdash; for creating atom groups.

**Perform the analysis of your choice on either of the systems studied today.**

Possible examples:

- Calculate the concentration of NaCl solution we have, its density, and try to google the experimental data to compare with.
- Calculate how the self-diffusion coefficient of lipids or ions changes upon aggregation/dissolution.
- ...

**Creativity is appreciated. :)**